# Project 1 — Exploratory Data Analysis
Run the cells **in order**, top to bottom. Only NumPy + matplotlib are used (allowed for this project).

Plan:
1. Load the data
2. Label balance
3. Missing values
4. Feature types
5. BRFSS special codes → clean them
6. Which features relate to the label?
7. Explore any single feature
8. Redundancy between features
9. Export a per-feature summary to split the work in the team

## 0. Setup

In [ ]:
import os, io, re
import numpy as np
import matplotlib.pyplot as plt

# Consistent colors: blue = no heart disease (-1), orange = heart disease (+1)
NEG, POS, GREY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({
    "figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True, "font.size": 11,
})
import zipfile, glob

# Paths are relative to this notebook (eda/). Change DATA_ZIP if your zip lives elsewhere.
DATA_ZIP     = "../data/dataset_to_release.zip"
EXTRACT_DIR  = "../data/dataset_unzipped"        # the zip is unpacked here (once)
CACHE_DIR    = "../data/cache"      # fast .npy copies of the CSVs

if not glob.glob(os.path.join(EXTRACT_DIR, "**", "x_train.csv"), recursive=True):
    print("Unzipping", DATA_ZIP, "...")
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(EXTRACT_DIR)

# find the folder that contains the CSVs, wherever it is inside the zip
DATA_DIR = os.path.dirname(glob.glob(os.path.join(EXTRACT_DIR, "**", "x_train.csv"), recursive=True)[0])
os.makedirs(CACHE_DIR, exist_ok=True)
print("CSV folder:", DATA_DIR)

## 1. Load the data
The first run unzips `data/dataset_to_release.zip`, parses the CSVs (~2 min) and saves `.npy` copies in `data/cache/`; later runs take a few seconds.
(`data/dataset_unzipped/` and `data/cache/` are big — add them to `.gitignore`.)

In [ ]:
def load(name):
    """Return (header, array) for a CSV, using a .npy cache. Empty cells become NaN."""
    path = os.path.join(DATA_DIR, name)
    cache = os.path.join(CACHE_DIR, name.replace(".csv", ".npy"))
    with open(path) as f:
        header = f.readline().strip().split(",")
        if os.path.exists(cache):
            return header, np.load(cache)
        txt = f.read()
    txt = re.sub(r"(?<=,)(?=,|\n)", "nan", txt)   # empty cell -> nan
    arr = np.loadtxt(io.StringIO(txt), delimiter=",")
    np.save(cache, arr)
    return header, arr

hx, Xtr_raw = load("x_train.csv")
_,  Ytr     = load("y_train.csv")
_,  Xte_raw = load("x_test.csv")

names   = hx[1:]                  # feature names (drop "Id")
X_raw   = Xtr_raw[:, 1:].astype(np.float32)   # train features (float32 halves memory)
XT_raw  = Xte_raw[:, 1:].astype(np.float32)   # test features
del Xtr_raw, Xte_raw
y       = Ytr[:, 1].astype(int)   # labels in {-1, +1}
idx     = {n: i for i, n in enumerate(names)}   # name -> column index

print("train:", X_raw.shape, " test:", XT_raw.shape, " labels:", np.unique(y))

## 2. Label balance

In [ ]:
pos_rate = (y == 1).mean()
fig, ax = plt.subplots(figsize=(6, 3))
counts = [(y == -1).sum(), (y == 1).sum()]
bars = ax.barh(["no MICHD (-1)", "MICHD (+1)"], counts, color=[NEG, POS], height=0.5)
for b, c in zip(bars, counts):
    ax.text(b.get_width(), b.get_y() + b.get_height()/2, f"  {c:,} ({c/len(y):.1%})", va="center")
ax.set_xlabel("number of respondents"); ax.set_title("Training labels are heavily imbalanced")
ax.set_xlim(0, max(counts) * 1.25); plt.show()

print(f"Always predicting -1 -> accuracy {1-pos_rate:.1%}, but F1 = 0. Evaluate with F1!")

## 3. Missing values
How empty is each feature? And is the test set missing the same things as the train set?

In [ ]:
miss_tr = np.isnan(X_raw).mean(0)
miss_te = np.isnan(XT_raw).mean(0)

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(np.sort(miss_tr)[::-1], color=NEG, lw=2)
for t in [0.5, 0.9]:
    axs[0].axhline(t, color=GREY, ls="--", lw=1)
    axs[0].text(len(names), t, f" {(miss_tr > t).sum()} features > {t:.0%}", va="bottom", ha="right")
axs[0].set_xlabel("features (sorted)"); axs[0].set_ylabel("fraction missing")
axs[0].set_title("Missing fraction per feature (train)")

axs[1].scatter(miss_tr, miss_te, s=10, color=NEG, alpha=0.6)
axs[1].plot([0, 1], [0, 1], color=GREY, ls="--", lw=1)
axs[1].set_xlabel("missing fraction — train"); axs[1].set_ylabel("missing fraction — test")
axs[1].set_title("Train vs test missingness (on the line = same)")
plt.tight_layout(); plt.show()

print("rows with no missing value:", (~np.isnan(X_raw).any(1)).sum())

Does **being missing** itself say something about the label? (e.g. a question only asked to sick people)

In [ ]:
# heart-disease rate among rows where the feature is missing vs observed
rate_missing, rate_observed = [], []
for j in range(X_raw.shape[1]):
    m = np.isnan(X_raw[:, j])
    rate_missing.append((y[m] == 1).mean() if m.any() else np.nan)
    rate_observed.append((y[~m] == 1).mean() if (~m).any() else np.nan)
gap = np.array(rate_missing) - np.array(rate_observed)

top = np.argsort(-np.nan_to_num(np.abs(gap)))[:15]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([names[j] for j in top][::-1], gap[top][::-1], color=[POS if g > 0 else NEG for g in gap[top][::-1]])
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("MICHD rate when missing  −  MICHD rate when observed")
ax.set_title("Features whose missingness is informative (→ a 'was missing' flag may help)")
plt.tight_layout(); plt.show()

## 4. Feature types
A rough hint from the number of distinct values. Always confirm with the BRFSS 2015 codebook.

In [ ]:
nuniq = np.array([len(np.unique(c[~np.isnan(c)])) for c in X_raw.T])
kind = np.where(nuniq <= 2, "binary", np.where(nuniq <= 10, "categorical", "continuous / coded"))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(np.log10(np.maximum(nuniq, 1)), bins=40, color=NEG, edgecolor="white")
ax.set_xlabel("log10(# distinct values)"); ax.set_ylabel("# features")
ax.set_title("Most features take only a handful of values → categorical")
plt.show()
for k in ["binary", "categorical", "continuous / coded"]:
    print(f"{k:20s}: {(kind == k).sum()} features")

## 5. BRFSS special codes
BRFSS stores non-answers as numbers, which would fool any model:

| code | meaning | fix |
|---|---|---|
| 7, 77, 777, 7777 | don't know / not sure | → NaN |
| 9, 99, 999, 9999 | refused | → NaN |
| 88, 888 | "none" (e.g. 0 bad-health days) | → 0 |
| 1 / 2 | yes / no | → 1 / 0 |

Look at a few raw examples first:

In [ ]:
def value_counts(col, top=10):
    v, k = np.unique(col[~np.isnan(col)], return_counts=True)
    o = np.argsort(-k)[:top]
    return ", ".join(f"{v[i]:g}:{k[i]}" for i in sorted(o))

for n in ["GENHLTH", "PHYSHLTH", "SMOKE100", "WEIGHT2", "CHILDREN", "_AGE80"]:
    print(f"{n:9s} ->", value_counts(X_raw[:, idx[n]]))

A **first-pass automatic cleaner**. It guesses the code scheme from each column's max value.
⚠️ It is a heuristic — check every column you own against the codebook and add exceptions in `MANUAL`.
Columns starting with `_` are already computed by the CDC and are left mostly as-is.

In [ ]:
MANUAL = {
    # "COLNAME": {"nan": [...], "zero": [...]},   <- add codebook-based fixes here
    # e.g. WEIGHT2: 9000-9998 means weight in kg (+9000) -> needs a conversion, not just NaN
}

def clean_column(name, col):
    col = col.copy()
    rule = MANUAL.get(name)
    if rule is None:
        mx = np.nanmax(col) if (~np.isnan(col)).any() else 0
        if   mx in (7, 9):          rule = {"nan": [7, 9],        "zero": []}
        elif mx in (77, 99):        rule = {"nan": [77, 99],      "zero": [88]}
        elif mx in (777, 999):      rule = {"nan": [777, 999],    "zero": [888]}
        elif mx in (7777, 9999):    rule = {"nan": [7777, 9999],  "zero": []}
        else:                       rule = {"nan": [],            "zero": []}
        if name.startswith("_"):    rule = {"nan": [v for v in rule["nan"] if v in (9, 99, 999, 9999)] if mx in (9, 99, 999, 9999) else [], "zero": []}
    col[np.isin(col, rule["nan"])] = np.nan
    col[np.isin(col, rule["zero"])] = 0
    return col

X  = np.column_stack([clean_column(n, X_raw[:, j])  for j, n in enumerate(names)])
XT = np.column_stack([clean_column(n, XT_raw[:, j]) for j, n in enumerate(names)])

for n in ["GENHLTH", "PHYSHLTH", "SMOKE100", "WEIGHT2"]:
    print(f"{n:9s} before ->", value_counts(X_raw[:, idx[n]], 8))
    print(f"{'':9s} after  ->", value_counts(X[:, idx[n]], 8), "\n")

## 6. Which features relate to the label?
Correlation with the label, computed on the **cleaned** data (NaN filled with the column mean, just for this look).

In [ ]:
def corr_with_label(M, y):
    """Pearson correlation of each column with y (column by column to save memory)."""
    yc = (y - y.mean()) / y.std()
    c = np.zeros(M.shape[1])
    for j in range(M.shape[1]):
        col = M[:, j].astype(np.float64)
        col = np.where(np.isnan(col), np.nanmean(col) if (~np.isnan(col)).any() else 0, col)
        if col.std() > 0:
            c[j] = ((col - col.mean()) / col.std() * yc).mean()
    return c

corr = corr_with_label(X, y)
top = np.argsort(-np.abs(corr))[:25]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh([names[j] for j in top][::-1], corr[top][::-1],
        color=[POS if c > 0 else NEG for c in corr[top][::-1]])
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("correlation with label (+ = more MICHD)")
ax.set_title("Top 25 features by |correlation| with heart disease")
plt.tight_layout(); plt.show()
print("Remember: yes/no questions are coded 1=yes, 2=no, so a NEGATIVE sign often means 'yes → more risk'.")

## 7. Explore a single feature
Change the name and re-run. Bars = how many people per value; orange line = % with heart disease at that value.

In [ ]:
def plot_feature(name, bins=20, use_clean=True):
    col = (X if use_clean else X_raw)[:, idx[name]]
    ok = ~np.isnan(col); c, yy = col[ok], y[ok]
    vals = np.unique(c)
    if len(vals) <= 25:                          # categorical: one bar per value
        keys = vals; groups = [c == v for v in vals]; labels = [f"{v:g}" for v in vals]
    else:                                        # continuous: quantile bins
        edges = np.unique(np.quantile(c, np.linspace(0, 1, bins + 1)))
        b = np.clip(np.searchsorted(edges, c, side="right") - 1, 0, len(edges) - 2)
        groups = [b == i for i in range(len(edges) - 1)]
        labels = [f"{edges[i]:.3g}" for i in range(len(edges) - 1)]
    n = np.array([g.sum() for g in groups]); r = np.array([(yy[g] == 1).mean() if g.any() else np.nan for g in groups])

    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    x = np.arange(len(labels)); rot = 45 if len(x) > 10 else 0
    ax[0].bar(x, n, color=NEG)
    ax[0].set_xticks(x); ax[0].set_xticklabels(labels, rotation=rot, fontsize=8)
    ax[0].set_ylabel("# respondents"); ax[0].set_title(f"{name}: how many people per value")

    ax[1].plot(x, r * 100, color=POS, marker="o", lw=2)
    ax[1].axhline(pos_rate * 100, color=GREY, ls="--", lw=1, label=f"overall {pos_rate:.1%}")
    ax[1].set_xticks(x); ax[1].set_xticklabels(labels, rotation=rot, fontsize=8)
    ax[1].set_ylabel("% with MICHD"); ax[1].set_title(f"{name}: MICHD rate per value"); ax[1].legend()

    ax[2].hist([c[yy == -1], c[yy == 1]], bins=min(30, len(vals)), density=True,
               color=[NEG, POS], label=["no MICHD", "MICHD"], edgecolor="white")
    ax[2].set_title(f"{name}: distribution by class"); ax[2].legend()
    plt.tight_layout(); plt.show()
    print(f"{name}: missing {np.isnan(col).mean():.1%} | distinct values {len(vals)} | corr {corr[idx[name]]:+.3f}")

plot_feature("GENHLTH")

In [ ]:
# the top-6 correlated features, one after another
for j in top[:6]:
    plot_feature(names[j])

In [ ]:
# continuous features worth a look: age, BMI, height, weight (cleaned)
for n in ["_AGE80", "_BMI5", "HTM4", "WTKG3"]:
    if n in idx: plot_feature(n)

## 8. Redundancy between features
Many features are near-copies (e.g. age in 4 versions). Highly correlated pairs can be merged or dropped.

In [ ]:
k = 25
sel = top[:k]
Mf = np.where(np.isnan(X[:, sel]), np.nanmean(X[:, sel], 0), X[:, sel])
C = np.corrcoef(Mf, rowvar=False)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(k)); ax.set_xticklabels([names[j] for j in sel], rotation=90, fontsize=8)
ax.set_yticks(range(k)); ax.set_yticklabels([names[j] for j in sel], fontsize=8)
ax.grid(False); plt.colorbar(im, fraction=0.046, label="correlation")
ax.set_title("Correlation among the top-25 features"); plt.tight_layout(); plt.show()

pairs = [(abs(C[a, b]), names[sel[a]], names[sel[b]]) for a in range(k) for b in range(a + 1, k) if abs(C[a, b]) > 0.8]
print("near-duplicate pairs (|corr| > 0.8):")
for c, a, b in sorted(pairs, reverse=True): print(f"  {a:10s} ~ {b:10s}  {c:.2f}")

## 9. Per-feature summary → split the columns between the 3 of us
Exports `feature_summary.csv` (open it in Excel/Sheets). Each person fills `type`, `keep`, `notes` for their third using the codebook.

In [ ]:
owners = np.array(["A", "B", "C"])[np.minimum(np.arange(len(names)) * 3 // len(names), 2)]  # contiguous thirds

with open("feature_summary.csv", "w") as f:
    f.write("col,name,owner,missing_train,missing_test,n_distinct,guess_type,corr_label,type,keep,notes\n")
    for j, n in enumerate(names):
        f.write(f"{j},{n},{owners[j]},{miss_tr[j]:.3f},{miss_te[j]:.3f},{nuniq[j]},{kind[j]},{corr[j]:+.4f},,,\n")

for o in "ABC":
    cols = np.where(owners == o)[0]
    print(f"{o}: columns {cols[0]}–{cols[-1]}  ({names[cols[0]]} … {names[cols[-1]]}), {len(cols)} features")

## Takeaways to fill in together
- Evaluation metric: **F1** (only 8.8% positives).
- Missing values: …
- Special codes cleaned: …
- Most promising features: …
- Features to drop / merge: …